In [ ]:
import os
import subprocess
import shutil
import glob
import random
import pandas as pd
from dotenv import load_dotenv

# === CONFIG ===
BASE_DIR = os.getcwd()
CLONED_DIR = os.path.join(BASE_DIR, "Cloned_Repos")
CONFIG_DIR = os.path.join(BASE_DIR, "Config_Files")
BUILDINFO_DIR = os.path.join(BASE_DIR, "BuildInfo")
EXTRA_DIR = os.path.join(BASE_DIR, "Extra_Files")

os.makedirs(CLONED_DIR, exist_ok=True)
os.makedirs(CONFIG_DIR, exist_ok=True)
os.makedirs(BUILDINFO_DIR, exist_ok=True)
os.makedirs(EXTRA_DIR, exist_ok=True)

# === Load START_NUMBER from .env ===
load_dotenv("All_Tokens.env")
START_NUMBER = int(os.getenv("START_NUMBER", "0"))
END_NUMBER = START_NUMBER + 50  # or any chunk size you prefer

# === STOP/RUN FLAG ===
RUN = True  # set False to stop immediately

# === Test keywords ===
test_keywords = ["androidtest", "espresso", "unit_test", "unittest", "junit"]

# === Load repo list ===
repo_df = pd.read_csv("Your_Repo_List.csv")  # adjust to your CSV
repo_urls = repo_df["clone_url"].tolist()

# === Filter by range ===
repo_urls = repo_urls[START_NUMBER:END_NUMBER]

# === Track cloned repos ===
cloned_repos = []

# === Clone & extract loop ===
for url in repo_urls:
    if not RUN:
        print("⏸️  Script stopped by RUN flag.")
        break

    repo_name = url.split("/")[-1].replace(".git", "")
    dest = os.path.join(CLONED_DIR, repo_name)

    if not os.path.exists(dest):
        subprocess.run(["git", "clone", "--depth", "1", url, dest])

    # === 1) Save all .yml/.yaml ===
    ymls = glob.glob(os.path.join(dest, ".github", "workflows", "*.yml"))
    yamls = glob.glob(os.path.join(dest, ".github", "workflows", "*.yaml"))
    for f in ymls + yamls:
        target_dir = os.path.join(CONFIG_DIR, repo_name)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy2(f, target_dir)

    # === 2) Save ALL build.gradle and build.gradle.kts ===
    gradle_files = glob.glob(os.path.join(dest, "**", "build.gradle"), recursive=True)
    gradle_kts_files = glob.glob(os.path.join(dest, "**", "build.gradle.kts"), recursive=True)
    for f in gradle_files + gradle_kts_files:
        target_dir = os.path.join(BUILDINFO_DIR, repo_name)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy2(f, target_dir)

    # === 3) Save .sh/.json if test keywords ===
    sh_files = glob.glob(os.path.join(dest, "**", "*.sh"), recursive=True)
    json_files = glob.glob(os.path.join(dest, "**", "*.json"), recursive=True)
    for f in sh_files + json_files:
        try:
            with open(f, "r", encoding="utf-8") as file:
                content = file.read().lower()
                if any(kw in content for kw in test_keywords):
                    target_dir = os.path.join(EXTRA_DIR, repo_name)
                    os.makedirs(target_dir, exist_ok=True)
                    shutil.copy2(f, target_dir)
        except Exception as e:
            print(f"⚠️ Error reading {f}: {e}")

    cloned_repos.append(repo_name)

print(f"✅ Finished cloning & extracting for {len(cloned_repos)} repos.")

# === 4) Random sample: keep 150 full repos, delete the rest ===
if len(cloned_repos) > 150:
    keep_repos = random.sample(cloned_repos, 150)
else:
    keep_repos = cloned_repos

for repo_name in cloned_repos:
    if repo_name not in keep_repos:
        repo_path = os.path.join(CLONED_DIR, repo_name)
        shutil.rmtree(repo_path, ignore_errors=True)

print(f"✅ Kept {len(keep_repos)} full repos, deleted the rest.")

# === 5) Save summary ===
summary_df = pd.DataFrame({
    "repo": cloned_repos,
    "kept_full": [r in keep_repos for r in cloned_repos]
})
summary_df.to_csv("Cloned_Repos_Summary.csv", index=False)
print("✅ Summary saved to Cloned_Repos_Summary.csv")
